# Day 34 Tutorial：MUTAG、GIN 与整图分类

> 这是公开小数据上的教学实验，不是粘合剂实验，也不是正式 MUTAG 基准。

## Goal

理解多张图怎样组成 batch，检查 `GINConv → global_add_pool → logits` 的 shape，并在封存 test 的前提下比较训练集多数类基线与固定轮数 GIN 的 validation accuracy。

## Setup

首次运行需要联网下载 MUTAG。缓存只写入仓库根目录 `.cache/pyg/`，不进入 Git。固定随机划分只用于教学；本 Notebook 不创建 test loader，也不计算 test accuracy。

In [1]:
from collections import Counter
from pathlib import Path
import random
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")

import numpy as np
import torch
from torch.nn import functional as F
from torch.utils.data import Subset
from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, global_add_pool

SPLIT_SEED = 34
MODEL_SEED = 3401
HIDDEN_CHANNELS = 32
BATCH_SIZE = 32
EPOCHS = 60
LEARNING_RATE = 0.01

def find_repo_root(start=Path.cwd().resolve()):
    for candidate in (start, *start.parents):
        if (candidate / "curriculum").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("没有找到 ML-practice 仓库根目录")

REPO_ROOT = find_repo_root()
CACHE_ROOT = REPO_ROOT / ".cache" / "pyg"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

print("torch:", torch.__version__)
print("PyG cache: repository .cache/pyg (not tracked)")

torch: 2.13.0
PyG cache: repository .cache/pyg (not tracked)


## Steps

### 1. 读取公开 MUTAG 并检查单张图

官方数据站记录 MUTAG 有 188 张图和 2 类。`edge_index` 对无向边保存两个方向，因此列数不是不重复无向边数。

In [2]:
dataset = TUDataset(root=str(CACHE_ROOT), name="MUTAG")
first_graph = dataset[0]

print("graphs:", len(dataset))
print("node features:", dataset.num_features)
print("classes:", dataset.num_classes)
print("first x:", tuple(first_graph.x.shape))
print("first edge_index:", tuple(first_graph.edge_index.shape))
print("first y:", tuple(first_graph.y.shape))

graphs: 188
node features: 7
classes: 2
first x: (17, 7)
first edge_index: (2, 38)
first y: (1,)


### 2. 只按索引建立固定 60/20/20 划分

打乱索引时不读取标签。训练和 validation 标签用于各自的训练/开发任务；test 索引只被保存和做集合互斥检查。单次随机划分不是论文级评价。

In [3]:
indices = list(range(len(dataset)))
random.Random(SPLIT_SEED).shuffle(indices)

n_train = int(0.60 * len(indices))
n_valid = int(0.20 * len(indices))
train_indices = indices[:n_train]
valid_indices = indices[n_train:n_train + n_valid]
test_indices = indices[n_train + n_valid:]

train_subset = Subset(dataset, train_indices)
valid_subset = Subset(dataset, valid_indices)

print("split sizes:", len(train_indices), len(valid_indices), len(test_indices))
print("train class counts:", dict(Counter(int(dataset[i].y.item()) for i in train_indices)))
print("validation class counts:", dict(Counter(int(dataset[i].y.item()) for i in valid_indices)))
print("test labels remain sealed")

split sizes: 112 37 39
train class counts: {1: 77, 0: 35}
validation class counts: {1: 25, 0: 12}
test labels remain sealed


### 3. 检查图 batch，并计算多数类 validation 基线

`batch.batch` 是一行一个节点的图编号；`batch.y` 是一行一个图的标签。多数类只能由训练标签决定。

In [4]:
loader_generator = torch.Generator().manual_seed(MODEL_SEED)
train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=loader_generator,
)
valid_loader = DataLoader(valid_subset, batch_size=BATCH_SIZE, shuffle=False)
one_batch = next(iter(valid_loader))

train_labels = [int(dataset[i].y.item()) for i in train_indices]
valid_labels = [int(dataset[i].y.item()) for i in valid_indices]
majority_class = Counter(train_labels).most_common(1)[0][0]
majority_valid_accuracy = sum(
    label == majority_class for label in valid_labels
) / len(valid_labels)

print("batch graphs:", one_batch.num_graphs)
print("batch x:", tuple(one_batch.x.shape))
print("batch edge_index:", tuple(one_batch.edge_index.shape))
print("batch vector:", tuple(one_batch.batch.shape))
print("batch y:", tuple(one_batch.y.shape))
print("training-majority validation accuracy:", round(majority_valid_accuracy, 4))

batch graphs: 32
batch x: (616, 7)
batch edge_index: (2, 1368)
batch vector: (616,)
batch y: (32,)
training-majority validation accuracy: 0.6757


### 4. 建立两层 GIN，逐步检查节点、图和 logits shape

In [5]:
def make_mlp(in_channels, hidden_channels):
    return torch.nn.Sequential(
        torch.nn.Linear(in_channels, hidden_channels),
        torch.nn.ReLU(),
        torch.nn.Linear(hidden_channels, hidden_channels),
    )

class SmallGIN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GINConv(make_mlp(in_channels, hidden_channels))
        self.conv2 = GINConv(make_mlp(hidden_channels, hidden_channels))
        self.output = torch.nn.Linear(hidden_channels, out_channels)

    def encode(self, x, edge_index, batch_vector):
        node_hidden = torch.relu(self.conv1(x, edge_index))
        node_hidden = torch.relu(self.conv2(node_hidden, edge_index))
        graph_hidden = global_add_pool(node_hidden, batch_vector)
        return node_hidden, graph_hidden

    def forward(self, x, edge_index, batch_vector):
        _, graph_hidden = self.encode(x, edge_index, batch_vector)
        return self.output(graph_hidden)

torch.manual_seed(MODEL_SEED)
shape_model = SmallGIN(
    dataset.num_features,
    HIDDEN_CHANNELS,
    dataset.num_classes,
)
with torch.no_grad():
    node_hidden, graph_hidden = shape_model.encode(
        one_batch.x,
        one_batch.edge_index,
        one_batch.batch,
    )
    initial_logits = shape_model(
        one_batch.x,
        one_batch.edge_index,
        one_batch.batch,
    )

print("node hidden:", tuple(node_hidden.shape))
print("graph hidden:", tuple(graph_hidden.shape))
print("logits:", tuple(initial_logits.shape))

node hidden: (616, 32)
graph hidden: (32, 32)
logits: (32, 2)


### 5. 固定 60 个 epoch 训练，只评价 validation

本教程不做 early stopping，不按 validation 峰值保存 checkpoint，也不读取 test 标签。

In [6]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def loader_accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for graph_batch in loader:
            logits = model(
                graph_batch.x,
                graph_batch.edge_index,
                graph_batch.batch,
            )
            labels = graph_batch.y.view(-1)
            correct += int((logits.argmax(dim=1) == labels).sum())
            total += labels.numel()
    return correct / total

set_seed(MODEL_SEED)
model = SmallGIN(
    dataset.num_features,
    HIDDEN_CHANNELS,
    dataset.num_classes,
)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
training_loss = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    loss_sum = 0.0
    graph_count = 0
    for graph_batch in train_loader:
        optimizer.zero_grad()
        logits = model(
            graph_batch.x,
            graph_batch.edge_index,
            graph_batch.batch,
        )
        labels = graph_batch.y.view(-1)
        loss = F.cross_entropy(logits, labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * labels.numel()
        graph_count += labels.numel()
    training_loss.append(loss_sum / graph_count)

gin_valid_accuracy = loader_accuracy(model, valid_loader)
print("first/final training loss:", round(training_loss[0], 4), round(training_loss[-1], 4))
print("majority validation accuracy:", round(majority_valid_accuracy, 4))
print("fixed-epoch GIN validation accuracy:", round(gin_valid_accuracy, 4))
print("test accuracy: sealed and not computed")

first/final training loss: 0.8228 0.3504
majority validation accuracy: 0.6757
fixed-epoch GIN validation accuracy: 0.7568
test accuracy: sealed and not computed


## Checks

断言只检查数据流、划分互斥和有限数值，不把 validation 结果包装成正式模型结论。

In [7]:
train_set = set(train_indices)
valid_set = set(valid_indices)
test_set = set(test_indices)

assert len(dataset) == 188
assert dataset.num_classes == 2
assert train_set.isdisjoint(valid_set)
assert train_set.isdisjoint(test_set)
assert valid_set.isdisjoint(test_set)
assert train_set | valid_set | test_set == set(range(len(dataset)))

assert one_batch.x.shape[0] == one_batch.batch.shape[0]
assert one_batch.y.view(-1).shape[0] == one_batch.num_graphs
assert node_hidden.shape == (one_batch.num_nodes, HIDDEN_CHANNELS)
assert graph_hidden.shape == (one_batch.num_graphs, HIDDEN_CHANNELS)
assert initial_logits.shape == (one_batch.num_graphs, dataset.num_classes)
assert len(training_loss) == EPOCHS
assert np.isfinite(training_loss).all()
assert 0.0 <= majority_valid_accuracy <= 1.0
assert 0.0 <= gin_valid_accuracy <= 1.0

print("Checks passed: batch、pooling、logits 与 split 均正确。")
print("边界：MUTAG validation 不是粘合剂结果，也不是最终 test 证据。")

Checks passed: batch、pooling、logits 与 split 均正确。
边界：MUTAG validation 不是粘合剂结果，也不是最终 test 证据。


## Next Steps

1. 不看答案完成 `03_exercises.md`；
2. 用自己的话解释 `batch` 和 pooling；
3. 不因一次 validation 数字宣布 GIN 胜出；
4. 进入 Day 35，读取实际粘合剂空白模板并做 GNN Go/No-Go，而不是虚构结构。